In [122]:
# Terminal
# !where pip
# !where python
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [123]:
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

In [3]:
data_train = pd.read_csv('Dataset/train.csv')
data_test = pd.read_csv('Dataset/test.csv')

Performing EDA.

In [65]:
# Listing all the columns in the datasets

data_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 57477 entries, 0 to 57476
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   id              57477 non-null  int64
 1   model_a         57477 non-null  str  
 2   model_b         57477 non-null  str  
 3   prompt          57477 non-null  str  
 4   response_a      57477 non-null  str  
 5   response_b      57477 non-null  str  
 6   winner_model_a  57477 non-null  int64
 7   winner_model_b  57477 non-null  int64
 8   winner_tie      57477 non-null  int64
dtypes: int64(4), str(5)
memory usage: 3.9 MB


In [10]:
# Shape of the datasets
print('Training data shape:', data_train.shape)
print('Test data shape:', data_test.shape)

Training data shape: (57477, 9)
Test data shape: (3, 4)


In [13]:
# Checking for missing values in the datasets
train_missing = data_train.isnull().sum()
test_missing = data_test.isnull().sum()

print('Missing values in training data:', train_missing.sum())
print('Missing values in test data:', test_missing.sum())

Missing values in training data: 0
Missing values in test data: 0


In [58]:
# Checking for recurring models in the data

print('Total distinct models in training data by model:', 
      data_train['model_a'].value_counts().sum(), data_train['model_b'].value_counts().sum())
print('\n')

print('Distinct models in training and testing data by model:')
print('Model A and B:', data_train['model_a'].nunique(), data_train['model_b'].nunique())

print('\n')

print('Recurring models in training data by model:')
print('Model A:', (data_train['model_a'].value_counts().sum())-data_train['model_a'].nunique())
print('Model B:', (data_train['model_b'].value_counts().sum())-data_train['model_b'].nunique())


Total distinct models in training data by model: 57477 57477


Distinct models in training and testing data by model:
Model A and B: 64 64


Recurring models in training data by model:
Model A: 57413
Model B: 57413


In [80]:
# Extracting the unique models from the training data
unique_models_a = data_train['model_a'].unique()
unique_models_b = data_train['model_b'].unique()

print('Are models in a and b are same:', set(unique_models_a) == set(unique_models_b))
print('\n')

print('Unique models in training data by model:')
unqiue_models = set(unique_models_a).union(set(unique_models_b))
unique_models = list(unqiue_models)
print(unique_models)


Are models in a and b are same: True


Unique models in training data by model:
['qwen1.5-72b-chat', 'stripedhyena-nous-7b', 'gpt-3.5-turbo-0125', 'dolphin-2.2.1-mistral-7b', 'openhermes-2.5-mistral-7b', 'gpt-4-0314', 'oasst-pythia-12b', 'claude-2.1', 'mpt-7b-chat', 'deepseek-llm-67b-chat', 'mistral-7b-instruct', 'claude-1', 'claude-instant-1', 'zephyr-7b-alpha', 'llama-2-13b-chat', 'llama-2-70b-chat', 'chatglm-6b', 'gpt-3.5-turbo-1106', 'qwen-14b-chat', 'mpt-30b-chat', 'guanaco-33b', 'nous-hermes-2-mixtral-8x7b-dpo', 'pplx-70b-online', 'gpt-4-0125-preview', 'qwen1.5-4b-chat', 'mistral-medium', 'wizardlm-70b', 'gpt-4-0613', 'gpt-4-1106-preview', 'gpt4all-13b-snoozy', 'llama-13b', 'mistral-7b-instruct-v0.2', 'pplx-7b-online', 'starling-lm-7b-alpha', 'stablelm-tuned-alpha-7b', 'vicuna-33b', 'llama-2-7b-chat', 'tulu-2-dpo-70b', 'gemini-pro', 'chatglm2-6b', 'openchat-3.5-0106', 'solar-10.7b-instruct-v1.0', 'gemini-pro-dev-api', 'chatglm3-6b', 'claude-2.0', 'gpt-3.5-turbo-0314', 'wizardlm-1

In [71]:
# Checking the distribution of the target variables in the training data
print('Distribution of target variable in training data:')

win_a = data_train['winner_model_a'].sum()
win_b = data_train['winner_model_b'].sum()
tie = data_train['winner_tie'].sum()

print('Proportion of wins for Model A:', win_a)
print('Proportion of wins for Model B:', win_b)
print('Proportion of ties:', tie)


target_distribution = data_train[['winner_model_a', 'winner_model_b', 'winner_tie']].value_counts(normalize=True)
print(target_distribution)

Distribution of target variable in training data:
Proportion of wins for Model A: 20064
Proportion of wins for Model B: 19652
Proportion of ties: 17761
winner_model_a  winner_model_b  winner_tie
1               0               0             0.349079
0               1               0             0.341911
                0               1             0.309011
Name: proportion, dtype: float64


In [115]:
# Creating a mini dataframe for unique models with respect to their win, lose and tie counts for train and test separately.

unique_models_distribution_in_train = pd.DataFrame(columns=['model', 'win_count', 'lose_count', 'tie_count'])

unique_models_distribution_in_train['model'] = unique_models

for model in unique_models:
    win_count = data_train[data_train['model_a'] == model]['winner_model_a'].sum() + data_train[data_train['model_b'] == model]['winner_model_b'].sum()
    lose_count = data_train[data_train['model_a'] == model]['winner_model_a'].sum() + data_train[data_train['model_b'] == model]['winner_model_b'].sum()
    tie_count = data_train[data_train['model_a'] == model]['winner_tie'].sum() + data_train[data_train['model_b'] == model]['winner_tie'].sum()
    unique_models_distribution_in_train.loc[unique_models_distribution_in_train['model'] == model, 'win_count'] = win_count
    unique_models_distribution_in_train.loc[unique_models_distribution_in_train['model'] == model, 'lose_count'] = lose_count
    unique_models_distribution_in_train.loc[unique_models_distribution_in_train['model'] == model, 'tie_count'] = tie_count
    
print('Unique models distribution in training data:')
print(unique_models_distribution_in_train)

Unique models distribution in training data:
                        model win_count lose_count tie_count
0            qwen1.5-72b-chat       215        215       181
1        stripedhyena-nous-7b       206        206       308
2          gpt-3.5-turbo-0125       268        268       285
3    dolphin-2.2.1-mistral-7b        95         95       133
4   openhermes-2.5-mistral-7b       280        280       343
..                        ...       ...        ...       ...
59                  vicuna-7b       448        448       510
60               openchat-3.5       473        473       566
61             fastchat-t5-3b       196        196       319
62     codellama-34b-instruct       427        427       447
63                yi-34b-chat       531        531       474

[64 rows x 4 columns]


In [113]:
# Checking the distribution of unique models in the training data
unique_model_distribution_in_train =   data_train['unique_models']
model_a_distribution = pd.Series(unique_model_distribution_in_train).value_counts()
model_b_distribution = pd.Series(unique_model_distribution_in_train).value_counts()

unique_model_distribution = pd.concat([model_a_distribution, model_b_distribution], axis=1, keys=['Model A', 'Model B'])
print('Distribution of unique models in training data:')
# print(unique_model_distribution)

plt.figure(figsize=(60, 10))
plt.subplot(1, 2, 1)
model_a_distribution.plot(kind='bar', title='Distribution of Model A')
plt.subplot(1, 2, 2)
model_b_distribution.plot(kind='bar', title='Distribution of Model B')


KeyError: 'unique_models'

In [4]:
print(data_train.info())

<class 'pandas.DataFrame'>
RangeIndex: 57477 entries, 0 to 57476
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   id              57477 non-null  int64
 1   model_a         57477 non-null  str  
 2   model_b         57477 non-null  str  
 3   prompt          57477 non-null  str  
 4   response_a      57477 non-null  str  
 5   response_b      57477 non-null  str  
 6   winner_model_a  57477 non-null  int64
 7   winner_model_b  57477 non-null  int64
 8   winner_tie      57477 non-null  int64
dtypes: int64(4), str(5)
memory usage: 3.9 MB
None


In [5]:
print(data_train.head())

       id             model_a              model_b  \
0   30192  gpt-4-1106-preview           gpt-4-0613   
1   53567           koala-13b           gpt-4-0613   
2   65089  gpt-3.5-turbo-0613       mistral-medium   
3   96401    llama-2-13b-chat  mistral-7b-instruct   
4  198779           koala-13b   gpt-3.5-turbo-0314   

                                              prompt  \
0  ["Is it morally right to try to have a certain...   
1  ["What is the difference between marriage lice...   
2  ["explain function calling. how would you call...   
3  ["How can I create a test set for a very rare ...   
4  ["What is the best way to travel from Tel-Aviv...   

                                          response_a  \
0  ["The question of whether it is morally right ...   
1  ["A marriage license is a legal document that ...   
2  ["Function calling is the process of invoking ...   
3  ["Creating a test set for a very rare category...   
4  ["The best way to travel from Tel Aviv to Jeru...   

 

In [6]:
print( data_test.info())

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          3 non-null      int64
 1   prompt      3 non-null      str  
 2   response_a  3 non-null      str  
 3   response_b  3 non-null      str  
dtypes: int64(1), str(3)
memory usage: 228.0 bytes
None


In [11]:
print( data_test.head())

        id                                             prompt  \
0   136060  ["I have three oranges today, I ate an orange ...   
1   211333  ["You are a mediator in a heated political deb...   
2  1233961  ["How to initialize the classification head wh...   

                                          response_a  \
0                    ["You have two oranges today."]   
1  ["Thank you for sharing the details of the sit...   
2  ["When you want to initialize the classificati...   

                                          response_b  
0  ["You still have three oranges. Eating an oran...  
1  ["Mr Reddy and Ms Blue both have valid points ...  
2  ["To initialize the classification head when p...  


'''
Conditions & data patterns:
    len of the response effects the user preference (more or less).
    most winning model
    impact of tie count models
'''